<div align="center">

<a href="https://colab.research.google.com/github/utkukose/human-computer-interaction-NB-lecture/blob/main/exams/final/FIN2_conversational_interface.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Final option 2: A conversational interface

**Human-Computer Interaction (BIL-429 İnsan-Bilgisayar Etkileşimi)**<br>Final project: An interface with an artificial intelligence (AI) feature, designed and evaluated. Starter Colab notebook

[Project page](https://github.com/utkukose/human-computer-interaction-NB-lecture/tree/main/exams/final) | [Course repository](https://github.com/utkukose/human-computer-interaction-NB-lecture)

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

<sub>Content licensed under CC BY 4.0, code under MIT. This course is updated in line with current developments in the field. First version: November 2026.</sub>

</div>

## How to use this starter Colab notebook

Save a copy in Drive first, with File, Save a copy in Drive. Every code cell runs as it is with an example case: A chatbot that books a court at the sports centre of a university. Replace the values marked `# TODO` with the measurements of your own interface. Then run all cells again from the top, with Runtime, Run all, and use the printed values and figures in the report. Each section names the week and the lecture section that explain its method. The last section collects the numbers for the results table of the report. Keep your own data in the cells, so that the Colab notebook documents how every number of the report was obtained.

## 0. Setup

The cell loads the libraries and creates the dictionary `RESULTS`, which collects the numbers of the report.

In [ ]:
import math
import random
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
RESULTS = {}            # the numbers for the results table of the report
print("Setup complete.")

## 1. The dialogue manager as a tested state machine

*Lecture: [Week 14, Conversational interfaces: Turns, intents and slots](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-14/lecture.html#conversational-interfaces)*

The dialogue manager decides what the assistant says next. A table maps a state and an event to the next state, and a request for a person leads to the handover from any state. Tests run each list of events and compare the final state with the expected one. In the example, a chatbot books a court at the sports centre of a university. Write the states of your own assistant, and add a test for every path that matters.

In [ ]:
SLOTS = {   # TODO: the slots of your task and their values
    "sport": ["tennis", "squash", "badminton", "basketball"],
    "day": ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"],
    "time": ["09:00", "11:00", "13:00", "15:00", "17:00", "19:00"]}
FSM = {("start", "hello"): "ask_sport", ("ask_sport", "value"): "ask_day",
       ("ask_day", "value"): "ask_time", ("ask_time", "value"): "confirm",
       ("confirm", "yes"): "booked", ("confirm", "no"): "ask_sport"}
ANY = {"person": "handover"}         # events that work in every state

def step(state, event):
    """The next state; an event that the table does not know keeps the state."""
    if event in ANY:
        return ANY[event]
    return FSM.get((state, event), state)

def final_state(events, state="start"):
    for event in events:
        state = step(state, event)
    return state

CASES = [   # TODO: one case for every path that matters
    (["hello", "value", "value", "value", "yes"], "booked"),
    (["hello", "value", "value", "value", "no"], "ask_sport"),
    (["hello", "value", "person"], "handover"),
    (["hello", "cough", "value"], "ask_day"),
]

def test_fsm():
    for events, expected in CASES:
        got = final_state(events)
        assert got == expected, f"{events} -> {got}, expected {expected}"

test_fsm()
print(f"all {len(CASES)} cases passed")
RESULTS["tested paths of the dialogue manager"] = len(CASES)

## 2. When to ask back

*Lecture: [Week 14, Errors and confirmations](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-14/lecture.html#confirmation)*

The assistant mishears or misreads a value with the probability $e$. Without asking back, a task of $k$ slots is right with $(1 - e)^k$ in $k$ turns. Asking back every value takes $2k/(1 - e)$ turns. Asking back below a threshold $\theta$ of confidence takes $k\,T_1$ turns and is right with $P_1^k$, where $T_1 = \frac{r(2 - a) + e(2 - b)}{1 - e(1 - b)}$, $P_1 = \frac{r}{r + e\,b}$ and $r = 1 - e$. $a$ and $b$ are the shares of right and of wrong values whose confidence reaches $\theta$. Estimate $e$ and the ranges of the confidence from a test of at least 40 values with your assistant.

In [ ]:
E = 0.12              # TODO: share of the values that the assistant got wrong
RIGHT_C = (0.65, 1.0)   # TODO: range of the confidence of right values
WRONG_C = (0.30, 0.85)  # TODO: range of the confidence of wrong values
THETA, K = 0.8, len(SLOTS)

def shares(theta=THETA):
    """Shares of right (a) and wrong (b) values with a confidence >= theta."""
    a = (RIGHT_C[1] - max(theta, RIGHT_C[0])) / (RIGHT_C[1] - RIGHT_C[0])
    lo = min(max(theta, WRONG_C[0]), WRONG_C[1])
    b = (WRONG_C[1] - lo) / (WRONG_C[1] - WRONG_C[0])
    return a, b

def strategies(e=E, k=K, theta=THETA):
    """Turns and right tasks: no asking back, every value, below theta."""
    a, b = shares(theta)
    r = 1 - e
    t1 = (r * (2 - a) + e * (2 - b)) / (1 - e * (1 - b))
    p1 = r / (r + e * b)
    return {"none": (k, r ** k), "explicit": (2 * k / r, 1.0),
            "confidence": (k * t1, p1 ** k)}

def converse(strategy, e=E, theta=THETA, rng=random):
    """One simulated task: turns of the user and whether it ends right."""
    turns, right = 0, 0
    for _ in range(K):
        while True:
            wrong = rng.random() < e
            c = rng.uniform(*(WRONG_C if wrong else RIGHT_C))
            turns += 1
            ask = strategy == "explicit" or (strategy == "confidence" and c < theta)
            if not ask:
                right += not wrong
                break
            turns += 1                   # the answer to the question
            if not wrong:
                right += 1
                break
    return turns, right == K

rng = random.Random(RANDOM_STATE)
for name, (t, p) in strategies().items():
    sims = [converse(name, rng=rng) for _ in range(5000)]
    t_sim, p_sim = np.mean(sims, axis=0)
    print(f"{name:>10}: {t:.3f} turns, {p:.3f} right | simulated "
          f"{t_sim:.3f} turns, {p_sim:.3f} right")
    RESULTS[f"{name}: turns, right"] = f"{t:.3f}, {p:.3f}"

## 3. A test with users, evaluated by success and costs

*Lecture: [Week 14, Evaluating conversations](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-14/lecture.html#evaluating-conversations) and [Week 9, Completion rates](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-09/lecture.html#completion-rates)*

PARADISE (PARAdigm for DIalogue System Evaluation) explains the rated satisfaction of the users by the success of the task and by the costs of the dialogue. A linear regression of the user satisfaction (US) on normalised measures gives $\mathrm{US} = \beta_0 + \alpha\,\mathcal{N}(\text{success}) - w\,\mathcal{N}(\text{turns})$. Each participant does a few tasks with the assistant and rates the satisfaction from 1 to 7. The example simulates 30 sessions; a test with real participants asks for their consent and keeps the data anonymous.

In [ ]:
def adjusted_wald(x, n, z=1.96):
    """Adjusted Wald interval of a completion rate: (p_adj, low, high)."""
    p_adj = (x + z ** 2 / 2) / (n + z ** 2)
    margin = z * math.sqrt(p_adj * (1 - p_adj) / (n + z ** 2))
    return p_adj, max(0.0, p_adj - margin), min(1.0, p_adj + margin)

def z(x):
    """Normalise to mean 0 and standard deviation 1."""
    x = np.asarray(x, dtype=float)
    return (x - x.mean()) / x.std()

# TODO: replace the simulated sessions with yours: success 1 or 0, turns, rating
rng = np.random.default_rng(RANDOM_STATE)
success = (rng.random(30) < 0.8).astype(int)
turns = rng.integers(3, 11, 30)
rating = np.clip(np.round(4.2 + 0.9 * z(success) - 0.5 * z(turns)
                          + rng.normal(0, 0.5, 30)), 1, 7)
SESSIONS = pd.DataFrame({"success": success, "turns": turns, "rating": rating})
X = np.column_stack([np.ones(len(SESSIONS)), z(SESSIONS.success), z(SESSIONS.turns)])
beta, *_ = np.linalg.lstsq(X, SESSIONS.rating, rcond=None)
print(f"US = {beta[0]:.3f} + {beta[1]:.3f} N(success) - {-beta[2]:.3f} N(turns)")
x = int(SESSIONS.success.sum())
p, lo, hi = adjusted_wald(x, len(SESSIONS))
print(f"completed {x} of {len(SESSIONS)}: adjusted {p:.3f}, {lo:.3f} to {hi:.3f}")
RESULTS["weight of success"] = round(beta[1], 3)
RESULTS["weight of the turns"] = round(-beta[2], 3)
RESULTS["completion, 95 percent interval"] = f"{lo:.3f} to {hi:.3f}"

## 4. A review for dark patterns

*Lecture: [Week 14, Dark patterns](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-14/lecture.html#dark-patterns) and [Week 14, Privacy and consent](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-14/lecture.html#privacy-and-consent)*

Review the paths that matter to the users but not to the business: Refusing consent, unsubscribing and deleting the account. Count the taps of each path. With the keystroke-level model, a tap with a decision takes $t_M + t_P + t_B = 1.35 + 1.10 + 0.20 = 2.65$ s. Then check the five strategies of dark patterns of Gray and colleagues: Nagging, obstruction, sneaking, interface interference and forced action.

In [ ]:
TAP = 1.35 + 1.10 + 0.20       # M + P + B of the keystroke-level model (s)
PATHS = {   # TODO: taps of each path in your assistant
    "accept all notifications": 1, "refuse marketing": 4,
    "sign up for the newsletter": 1, "unsubscribe": 6, "delete the account": 9}
for name, taps in PATHS.items():
    word = "tap" if taps == 1 else "taps"
    print(f"{name:>27}: {taps} {word}, {taps * TAP:.2f} s")
CHECK = {   # TODO: found in your assistant? (True/False) and where
    "nagging": (True, "asks to rate the app after every booking"),
    "obstruction": (True, "unsubscribing takes six taps"),
    "sneaking": (False, ""),
    "interface interference": (True, "the refusal is a grey link"),
    "forced action": (False, "")}
for name, (found, where) in CHECK.items():
    print(f"{name:>23}: {'found' if found else 'not found'} {where}")
RESULTS["strategies of dark patterns found"] = sum(f for f, _ in CHECK.values())
RESULTS["slowest path"] = max(PATHS, key=PATHS.get)

## 5. Handing over to a person

*Lecture: [Week 14, Handing over to a person](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-14/lecture.html#handover)*

A rule hands a conversation over to a person when an urgent word appears or when the confidence of the assistant about the intent is below a threshold. Its recall is the share of the conversations that need a person and are handed over. Its precision is the share of the handed over that need a person. Label at least 30 conversations of your test: Did each one need a person?

In [ ]:
URGENT = ["injured", "bleeding", "fell", "complaint", "refund", "person"]   # TODO
LOG = [   # TODO: the message, the confidence about the intent, needs a person
    ("Book squash on Monday at 17:00", 0.93, 0),
    ("I fell on court 2 and my knee is bleeding", 0.71, 1),
    ("Can I talk to a person?", 0.55, 1),
    ("What does a court cost for students?", 0.88, 0),
    ("You charged me twice, I want a refund", 0.62, 1),
    ("Is the sports hall open on Sunday?", 0.91, 0),
    ("Move my tennis booking to Friday", 0.84, 0),
    ("The booking failed three times now", 0.47, 1),
    ("Where can I park at the sports centre?", 0.79, 0),
    ("Cancel my badminton booking", 0.90, 0),
    ("My card was blocked at the entrance", 0.52, 1),
    ("Can I bring a guest to basketball?", 0.74, 0),
]

def evaluate(theta, use_words=True):
    """Recall, precision and share handed over of the rule on the log."""
    tp = fp = fn = 0
    for text, c, need in LOG:
        hand = (use_words and any(w in text.lower() for w in URGENT)) or c < theta
        tp += hand and need
        fp += hand and not need
        fn += need and not hand
    return tp / max(tp + fn, 1), tp / max(tp + fp, 1), (tp + fp) / len(LOG)

for theta in (0.0, 0.5, 0.6, 0.7, 0.8):
    r, p, read = evaluate(theta)
    print(f"theta {theta}: recall {r:.3f}, precision {p:.3f}, "
          f"handed over {read:.3f}")
r, p, read = evaluate(0.6)      # TODO: the threshold of your design
RESULTS["handover: recall, precision"] = f"{r:.3f}, {p:.3f}"

## 6. Plain words in the prompts

*Lecture: [Week 10, Plain words](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-10/lecture.html#plain-words)*

A conversation is read sentence by sentence, and every prompt must be easy to read. The Flesch-Kincaid grade level estimates the years of school needed to read a text.

In [ ]:
def syllables_en(word):
    """Approximate English syllables: runs of vowels, minus a silent e."""
    w = re.sub(r"[^a-z]", "", word.lower())
    n = len(re.findall(r"[aeiouy]+", w))
    if n > 1 and w.endswith("e") and not w.endswith(("le", "ee")):
        n -= 1
    return max(1, n)

def fkgl(text):
    """Flesch-Kincaid grade level of an English text."""
    words = re.findall(r"[A-Za-z]+", text)
    sentences = [s for s in re.split(r"[.!?]+", text) if s.strip()]
    syll = sum(syllables_en(w) for w in words)
    return 0.39 * len(words) / len(sentences) + 11.8 * syll / len(words) - 15.59

PROMPTS = {   # TODO: the prompts of your assistant
    "greeting": "Hello! I am the booking assistant, a computer program. "
                "Which sport would you like to play?",
    "error": "Your reservation could not be processed due to an unexpected "
             "authentication discrepancy.",
}
for name, text in PROMPTS.items():
    print(f"{name}: grade {fkgl(text):.1f}")
RESULTS["highest grade of a prompt"] = round(
    max(fkgl(t) for t in PROMPTS.values()), 1)

## 7. Results for the report

The cell collects the numbers that the sections above stored in `RESULTS`. Copy them into the results table of the report, and explain each one in the text. A value that changes the design deserves a figure.

In [ ]:
import pandas as pd

print(pd.Series(RESULTS, name="value", dtype=object).to_string())

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

This course is updated in line with current developments in the field. First version: November 2026.

</div>